# Sea Surface Temperature Forecasting with FNO2D

Forecast future ocean surface temperature fields from past ones with a 2D Fourier Neural Operator
trained on NOAA Optimum Interpolation Sea Surface Temperature (OISST) data. The model learns the
operator

$$G: T(\cdot, t) \mapsto T(\cdot, t + \Delta t)$$

that advances the temperature field $T$ by one time step.

In [ ]:
# Install keras-sciml, then choose a backend: "tensorflow", "torch" or "jax"
!pip install keras-sciml
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

## Hyperparameters

`TIME_HISTORY` past fields are stacked as input channels and `TIME_FUTURE` fields are predicted. Set `USE_SYNTHETIC = True` to train on physically-motivated synthetic fields instead of downloading OISST data. `N_SAMPLES` and `RESOLUTION` can be set to subsample the data for a faster run.

In [2]:
N_SAMPLES = None
RESOLUTION = None
TIME_HISTORY = 1
TIME_FUTURE = 1
USE_SYNTHETIC = False
MODES = 8
WIDTH = 24
DEPTH = 3
EPOCHS = 25
BATCH_SIZE = 16
LEARNING_RATE = 0.005

## Load the data

`load_sea_surface_temperature` returns standardised train/test pairs. If the download fails, it falls
back to synthetic fields, and `meta["source"]` says which was used.

In [3]:
import keras
import numpy as np

from keras_sciml.datasets import load_sea_surface_temperature
from keras_sciml.losses import RelativeL2Loss, relative_l2_error
from keras_sciml.models import FNO2D

keras.utils.set_random_seed(42)

x_train, y_train, x_test, y_test, meta = load_sea_surface_temperature(
    download=True,
    n_samples=N_SAMPLES,
    resolution=RESOLUTION,
    time_history=TIME_HISTORY,
    time_future=TIME_FUTURE,
    step_mode="pairs",
    train_ratio=0.8,
    normalize=True,
    use_synthetic=USE_SYNTHETIC,
    seed=42,
)
print(f"Data source: {meta['source']}")
print(f"Train: {x_train.shape}, Test: {x_test.shape}")
print(f"Mean SST: {meta['mean']:.2f} deg C, Std: {meta['std']:.2f} deg C")

      0/9830528 ━━━━━━━━━━━━━━━━━━━━ 0s 0s/step

 270336/9830528 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step

 794624/9830528 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step

1646592/9830528 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

2572288/9830528 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

...

6086656/9830528 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

7077888/9830528 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

8404992/9830528 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

9830528/9830528 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Data source: noaa_oisst_sample
Train: (200, 64, 64, 1), Test: (50, 64, 64, 1)
Mean SST: 0.01 deg C, Std: 1.00 deg C


## Define the model

In [4]:
model = FNO2D(
    modes=(MODES, MODES),
    width=WIDTH,
    depth=DEPTH,
    out_channels=TIME_FUTURE,
    activation="gelu",
)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=RelativeL2Loss(),
    metrics=[RelativeL2Loss(name="rel_l2"), "mae"],
)

## Train

In [5]:
history = model.fit(
    x_train,
    y_train,
    validation_data=(x_test, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=2,
)

Epoch 1/25


13/13 - 11s - 878ms/step - loss: 0.6319 - mae: 0.4170 - rel_l2: 0.6207 - val_loss: 0.3281 - val_mae: 0.3130 - val_rel_l2: 0.3010


Epoch 2/25


13/13 - 5s - 354ms/step - loss: 0.3747 - mae: 0.2231 - rel_l2: 0.3706 - val_loss: 0.2897 - val_mae: 0.2750 - val_rel_l2: 0.2707


Epoch 3/25


...

13/13 - 5s - 368ms/step - loss: 0.1765 - mae: 0.1273 - rel_l2: 0.1750 - val_loss: 0.3162 - val_mae: 0.2911 - val_rel_l2: 0.2915


Epoch 24/25


13/13 - 5s - 354ms/step - loss: 0.1816 - mae: 0.1342 - rel_l2: 0.1796 - val_loss: 0.3220 - val_mae: 0.2992 - val_rel_l2: 0.2968


Epoch 25/25


13/13 - 4s - 314ms/step - loss: 0.1900 - mae: 0.1390 - rel_l2: 0.1875 - val_loss: 0.3218 - val_mae: 0.2968 - val_rel_l2: 0.2971


## Evaluate

The MAE is converted back to degrees Celsius using the standard deviation from standardisation.

In [6]:
predictions = keras.ops.convert_to_numpy(model.predict(x_test, verbose=0))
y_test_np = keras.ops.convert_to_numpy(y_test)

test_rel_l2 = float(np.mean(keras.ops.convert_to_numpy(relative_l2_error(y_test_np, predictions))))
mae_celsius = float(np.mean(np.abs(y_test_np - predictions)) * meta["std"])
print(f"Test relative L2 error: {test_rel_l2:.4f}")
print(f"Test mean absolute error: {mae_celsius:.3f} deg C")

Test relative L2 error: 0.3218
Test mean absolute error: 0.297 deg C
